# Brightline Logistics — Edge Model Trade-off Board (CST 627, Week 5)

Benchmarks small classifiers for in-cab tablets that read **scanned bills of lading** and **spoken driver notes**.
All logic lives in `scripts/`; this notebook only calls those scripts, in order.

| Step | Script |
|---|---|
| 1. Synthetic scanned documents (8 types, OCR noise, partial scans) | `generate_documents.py` |
| 2. Public driver notes (SLURP transcripts, 8 in-cab intents) | `fetch_driver_notes.py` |
| 3. Merge into one 16-class train / val / test set | `build_eval_set.py` |
| 4. Train a configuration | `train.py --config <name>` |
| 5. Benchmark it (warm-up, sync, fixed seeds, repeated trials → `results/runs.csv`) | `benchmark.py --config <name>` |
| 6. Build the Excel benchmark workbook (board, runs, training, scaling, recommendation) | `build_workbook.py` |

**Runtime:** Colab → *Runtime ▸ Change runtime type ▸ T4 GPU*.

## 0. Setup

In [1]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("csp-cst627-week5"):
        !git clone https://github.com/nmilleson/csp-cst627-week5.git
    %cd csp-cst627-week5
    !pip install -q -r requirements.txt
else:
    # Running locally from notebooks/: work from the repo root.
    if os.path.basename(os.getcwd()) == "notebooks":
        %cd ..

Cloning into 'csp-cst627-week5'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 16 (delta 0), reused 16 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 32.26 KiB | 3.58 MiB/s, done.
/content/csp-cst627-week5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00


In [2]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -c "import torch; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU available:', torch.cuda.is_available())"

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
torch 2.11.0+cu130 | CUDA 13.0 | GPU available: True


## 1–3. Build the evaluation set

* **Synthetic documents** — bill of lading, hazmat shipping paper, delivery receipt, OS&D exception report, freight invoice,
  fuel receipt, scale ticket, lumper receipt. Headers are often cut off, ~60% are partial photos, fields are dropped/shuffled,
  and OCR noise is 0–15% per document. The **test split uses company and city names never seen in training**.
* **Driver notes** — SLURP spoken-command transcripts (CC BY 4.0), filtered to 8 intents a driver would say in the cab;
  SLURP's own train/devel/test splits are kept.

In [3]:
!python scripts/generate_documents.py

train:  8000 documents -> /content/csp-cst627-week5/data/raw/synthetic_docs/train.jsonl
val  :  1000 documents -> /content/csp-cst627-week5/data/raw/synthetic_docs/val.jsonl
test :  1600 documents -> /content/csp-cst627-week5/data/raw/synthetic_docs/test.jsonl

Example (test):
------------------------------------------------------------
Beright1ine Lgistics | USDOT 3120457
Consignee: High Desert Provisions, 913 Commeree Dr, Amarilo, TX 68622
Bill To (hird Party): Timerlne Groers
PRO: 93-8s859186
Freight Terms: Prepaid
Ship Datc: 11/1S/26
26 DRUM  PVC ppefittings NMFC 072717 Class 55  2,314 lb
2 SKID  frozen pouYlty  NMFC 501020 Class 50  8,762 lbs
Page 1 of 1
------------------------------------------------------------
label: doc:bill_of_lading  meta: {'header_visible': False, 'cropped': True, 'ocr_noise': 0.0568, 'name_pool': 1}


In [4]:
!python scripts/fetch_driver_notes.py

downloading https://raw.githubusercontent.com/pswietojanski/slurp/master/dataset/slurp/train.jsonl
train:  3426 notes  {'calendar_set': 802, 'calendar_query': 558, 'email_sendemail': 347, 'email_query': 411, 'weather_query': 559, 'transport_query': 227, 'datetime_query': 340, 'alarm_set': 182}
downloading https://raw.githubusercontent.com/pswietojanski/slurp/master/dataset/slurp/devel.jsonl
val  :   615 notes  {'calendar_set': 130, 'calendar_query': 99, 'email_sendemail': 63, 'email_query': 72, 'weather_query': 122, 'transport_query': 36, 'datetime_query': 63, 'alarm_set': 30}
downloading https://raw.githubusercontent.com/pswietojanski/slurp/master/dataset/slurp/test.jsonl
test :   893 notes  {'calendar_set': 208, 'calendar_query': 126, 'email_sendemail': 113, 'email_query': 117, 'weather_query': 152, 'transport_query': 51, 'datetime_query': 85, 'alarm_set': 41}


In [5]:
!python scripts/build_eval_set.py

16 labels
train: 11426  slurp_notes=3426 (mean 38.6B, p95 69B, 0 truncated), synthetic_docs=8000 (mean 232.7B, p95 458B, 0 truncated)
val  :  1615  slurp_notes=615 (mean 38.5B, p95 69B, 0 truncated), synthetic_docs=1000 (mean 236.6B, p95 468B, 0 truncated)
test :  2493  slurp_notes=893 (mean 38.6B, p95 71B, 0 truncated), synthetic_docs=1600 (mean 231.0B, p95 458B, 0 truncated)


## 4–5. Baseline — 4-layer softmax-attention transformer (FP32)

In [6]:
!python scripts/train.py --config baseline

[baseline] 3,491,856 params (3,163,664 non-embedding) on cuda
  epoch  1  loss 1.3891  val_acc 0.7628
  epoch  2  loss 0.5150  val_acc 0.8204
  epoch  3  loss 0.3679  val_acc 0.8526
  epoch  4  loss 0.2883  val_acc 0.8749
  epoch  5  loss 0.2220  val_acc 0.8861
  epoch  6  loss 0.1735  val_acc 0.8898
  epoch  7  loss 0.1416  val_acc 0.8978
  epoch  8  loss 0.1239  val_acc 0.8947
best val_acc 0.8978  |  16,040,640 training tokens  |  168s  ->  /content/csp-cst627-week5/checkpoints/baseline.pt


In [7]:
!python scripts/benchmark.py --config baseline --trials 5 --warmup 50

[baseline] run eecf0fdd on Tesla T4 | 2493 requests x 5 trials, 50 warm-up each
  trial 0: acc 0.8813 (docs 0.96375, notes 0.73348)  p50 2.49 ms  p95 3.97 ms  peak 29.8 MB
  trial 1: acc 0.8813 (docs 0.96375, notes 0.73348)  p50 2.40 ms  p95 3.72 ms  peak 29.8 MB
  trial 2: acc 0.8813 (docs 0.96375, notes 0.73348)  p50 2.31 ms  p95 3.48 ms  peak 29.8 MB
  trial 3: acc 0.8813 (docs 0.96375, notes 0.73348)  p50 2.49 ms  p95 3.92 ms  peak 29.8 MB
  trial 4: acc 0.8813 (docs 0.96375, notes 0.73348)  p50 2.37 ms  p95 2.96 ms  peak 29.8 MB
median over 5 trials: acc 0.8813  p50 2.40 ms  p95 3.72 ms  peak 29.8 MB  ->  /content/csp-cst627-week5/results/runs.csv


## Variant A — architecture: linear attention
Same 4-layer model; softmax attention is replaced by linear (elu+1) attention, which scales O(T) instead of O(T²) in sequence length.

In [8]:
!python scripts/train.py --config linear_attention

[linear_attention] 3,491,856 params (3,163,664 non-embedding) on cuda
  epoch  1  loss 1.4293  val_acc 0.7040
  epoch  2  loss 0.6158  val_acc 0.7616
  epoch  3  loss 0.4887  val_acc 0.7969
  epoch  4  loss 0.4259  val_acc 0.8415
  epoch  5  loss 0.3691  val_acc 0.8526
  epoch  6  loss 0.3217  val_acc 0.8632
  epoch  7  loss 0.2946  val_acc 0.8582
  epoch  8  loss 0.2801  val_acc 0.8570
best val_acc 0.8632  |  16,040,640 training tokens  |  201s  ->  /content/csp-cst627-week5/checkpoints/linear_attention.pt


In [9]:
!python scripts/benchmark.py --config linear_attention --trials 5 --warmup 50

[linear_attention] run 1610f4f4 on Tesla T4 | 2493 requests x 5 trials, 50 warm-up each
  trial 0: acc 0.8712 (docs 0.965, notes 0.70325)  p50 3.98 ms  p95 6.08 ms  peak 29.8 MB
  trial 1: acc 0.8712 (docs 0.965, notes 0.70325)  p50 3.90 ms  p95 5.89 ms  peak 29.8 MB
  trial 2: acc 0.8712 (docs 0.965, notes 0.70325)  p50 3.80 ms  p95 5.32 ms  peak 29.8 MB
  trial 3: acc 0.8712 (docs 0.965, notes 0.70325)  p50 3.83 ms  p95 5.81 ms  peak 29.8 MB
  trial 4: acc 0.8712 (docs 0.965, notes 0.70325)  p50 3.86 ms  p95 6.08 ms  peak 29.8 MB
median over 5 trials: acc 0.8712  p50 3.86 ms  p95 5.89 ms  peak 29.8 MB  ->  /content/csp-cst627-week5/results/runs.csv


## Variant B — quantized: 4-bit NF4
No retraining: the baseline's linear layers are quantized post-training to 4-bit NF4 with bitsandbytes (FP16 matmuls). The classifier head stays FP32.

In [10]:
!python scripts/benchmark.py --config quant_4bit --trials 5 --warmup 50

[quant_4bit] run 3d0f40a4 on Tesla T4 | 2493 requests x 5 trials, 50 warm-up each
  trial 0: acc 0.8813 (docs 0.9625, notes 0.73572)  p50 5.03 ms  p95 7.35 ms  peak 19.4 MB
  trial 1: acc 0.8813 (docs 0.9625, notes 0.73572)  p50 5.18 ms  p95 7.54 ms  peak 19.4 MB
  trial 2: acc 0.8813 (docs 0.9625, notes 0.73572)  p50 4.91 ms  p95 7.27 ms  peak 19.4 MB
  trial 3: acc 0.8813 (docs 0.9625, notes 0.73572)  p50 4.96 ms  p95 7.20 ms  peak 19.4 MB
  trial 4: acc 0.8813 (docs 0.9625, notes 0.73572)  p50 5.08 ms  p95 7.27 ms  peak 19.4 MB
median over 5 trials: acc 0.8813  p50 5.03 ms  p95 7.27 ms  peak 19.4 MB  ->  /content/csp-cst627-week5/results/runs.csv


## Variant C — distilled student
A 2-layer, d=128 transformer trained on a mix of the baseline's temperature-softened predictions (α=0.7, T=2) and the hard labels.

In [11]:
!python scripts/train.py --config distilled_student

[distilled_student] 562,960 params (398,864 non-embedding) on cuda
  distilling from 'baseline' (alpha=0.7, T=2.0)
  epoch  1  loss 4.5477  val_acc 0.5678
  epoch  2  loss 1.5068  val_acc 0.7560
  epoch  3  loss 0.7860  val_acc 0.8161
  epoch  4  loss 0.5728  val_acc 0.8471
  epoch  5  loss 0.4451  val_acc 0.8632
  epoch  6  loss 0.3801  val_acc 0.8768
  epoch  7  loss 0.3237  val_acc 0.8854
  epoch  8  loss 0.2933  val_acc 0.8867
  epoch  9  loss 0.2683  val_acc 0.8954
  epoch 10  loss 0.2484  val_acc 0.8935
  epoch 11  loss 0.2384  val_acc 0.8972
  epoch 12  loss 0.2376  val_acc 0.8947
best val_acc 0.8972  |  24,060,960 training tokens  |  162s  ->  /content/csp-cst627-week5/checkpoints/distilled_student.pt


In [12]:
!python scripts/benchmark.py --config distilled_student --trials 5 --warmup 50

[distilled_student] run 6c3d1f09 on Tesla T4 | 2493 requests x 5 trials, 50 warm-up each
  trial 0: acc 0.8785 (docs 0.96312, notes 0.72676)  p50 1.54 ms  p95 2.11 ms  peak 14.9 MB
  trial 1: acc 0.8785 (docs 0.96312, notes 0.72676)  p50 1.52 ms  p95 2.09 ms  peak 14.9 MB
  trial 2: acc 0.8785 (docs 0.96312, notes 0.72676)  p50 1.84 ms  p95 2.54 ms  peak 14.9 MB
  trial 3: acc 0.8785 (docs 0.96312, notes 0.72676)  p50 1.55 ms  p95 2.10 ms  peak 14.9 MB
  trial 4: acc 0.8785 (docs 0.96312, notes 0.72676)  p50 1.57 ms  p95 2.35 ms  peak 14.9 MB
median over 5 trials: acc 0.8785  p50 1.55 ms  p95 2.11 ms  peak 14.9 MB  ->  /content/csp-cst627-week5/results/runs.csv


## Benchmark log (every trial of every run)

In [13]:
import pandas as pd
runs = pd.read_csv("results/runs.csv")
runs.groupby("config", sort=False)[["accuracy", "acc_synthetic_docs", "acc_slurp_notes", "p50_ms", "p95_ms",
                                    "peak_mem_mb", "checkpoint_mb", "params_total"]].mean().round(4)

,accuracy,acc_synthetic_docs,acc_slurp_notes,p50_ms,p95_ms,peak_mem_mb,checkpoint_mb,params_total
config,,,,,,,,
baseline,0.8813,0.9637,0.7335,2.4083,3.6090,29.78,13.34,3491856.0
linear_attention,0.8712,0.9650,0.7032,3.8717,5.8351,29.78,13.34,3491856.0
quant_4bit,0.8813,0.9625,0.7357,5.0330,7.3264,19.35,2.93,3491856.0
distilled_student,0.8785,0.9631,0.7268,1.6056,2.2384,14.95,2.16,562960.0


## Benchmark workbook
Builds `results/brightline_benchmark.xlsx`: **Board** (trade-off board with live formulas over every trial), **Runs**, **Training**,
**Scaling** (6ND + Chinchilla estimate for a 10x model), **Recommendation** (ranking + residual-risk register) and **Protocol**.

In [14]:
!python scripts/build_workbook.py

20 trial rows from 4 configs ['baseline', 'distilled_student', 'linear_attention', 'quant_4bit'] -> /content/csp-cst627-week5/results/brightline_benchmark.xlsx


In [15]:
# Download the workbook and raw logs (Colab storage is wiped when the runtime disconnects).
if IN_COLAB:
    !zip -qr results.zip results
    from google.colab import files
    files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>